# 03 · Force-field strain on the released conformers

MMFF94 strain in two flavours, reproducing `data/mmff94_strain_all.csv` for a few ligands (no model needed):

- **single point**: MMFF94 energies of the AIMNet2-CPCM bound and global conformers;
- **re-optimized**: bound conformer relaxed with rotatable-bond dihedrals restrained to their deposited values, global conformer relaxed freely, both on the MMFF94 surface.

Because the global endpoint is relaxed from the AIMNet2-CPCM minimum (not from an independent MMFF94 search) the re-optimized value is a *local* force-field strain.

In [1]:
from itertools import islice
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdForceFieldHelpers as FF, rdMolTransforms
from lcse import load_conformers, load_master_table
from lcse.strain import rotatable_dihedrals
df = load_master_table()

In [2]:
def mmff_energy(m, opt=False, restrain=None):
    m = Chem.Mol(m); p = FF.MMFFGetMoleculeProperties(m); ff = FF.MMFFGetMoleculeForceField(m, p)
    if opt:
        if restrain:
            c = m.GetConformer()
            for t in restrain:
                a = rdMolTransforms.GetDihedralDeg(c, *t); ff.MMFFAddTorsionConstraint(*t, False, a - 0.5, a + 0.5, 1000.0)
        ff.Initialize(); ff.Minimize(maxIts=2000)
        if restrain:
            return FF.MMFFGetMoleculeForceField(m, p).CalcEnergy()   # energy without the restraint term
    return ff.CalcEnergy()

In [3]:
N = 12
glob = {m.GetProp('_Name'): m for m in islice(load_conformers('global'), 3 * N)}
rows = []
for mb in islice(load_conformers('bound'), N):
    k = mb.GetProp('_Name'); mg = glob[k]
    sp = mmff_energy(mb) - mmff_energy(mg)
    ro = mmff_energy(mb, True, rotatable_dihedrals(mb)) - mmff_energy(mg, True)
    rows.append(dict(ligand=k, net_charge=df.loc[k, 'net_charge'], aimnet2_cpcm=df.loc[k, 'lcse_kcal'], mmff_sp=sp, mmff_reopt=ro,
                     table_sp=df.loc[k, 'mmff94_strain_sp_kcal'], table_reopt=df.loc[k, 'mmff94_strain_reopt_kcal']))
pd.DataFrame(rows).set_index('ligand').round(2)

,net_charge,aimnet2_cpcm,mmff_sp,mmff_reopt,table_sp,table_reopt
ligand,,,,,,
6SH_5LAR_A_401,0,7.16,8.06,17.82,8.06,17.82
6SL_5KGD_A_420,0,0.06,0.10,0.11,0.10,0.11
6SW_5KHH_A_701,-1,1.09,0.77,2.94,0.77,2.94
6SX_5KHI_A_701,-1,0.85,0.21,2.55,0.21,2.55
6SY_5KHJ_A_701,-1,1.27,0.71,1.91,0.71,1.91
6SZ_5KHK_A_701,-1,0.76,0.32,2.44,0.32,2.44
6TA_5KIT_B_501,0,4.63,-0.48,3.78,-0.48,3.78
6TD_5YA5_A_1401,0,2.05,2.00,1.75,2.00,1.75
6TF_5KJ2_A_1701,0,3.51,1.36,0.32,1.36,0.32


The `table_*` columns are the released values; they agree with the recomputation to numerical precision. Across all 7887 ligands (Table S4):

In [4]:
pd.read_csv('../data/mmff94_strain_summary_by_charge.csv', index_col=0).round(2)

,n,aimnet_med,mmff_sp_med,mmff_reopt_med,MAD_sp,MAD_reopt,bias_sp,bias_reopt,spearman_sp,spearman_reopt,neg_reopt_%
qcls,,,,,,,,,,,
-3,800.0,12.56,33.22,42.37,20.63,29.18,20.33,29.18,0.77,0.75,0.62
-2,1070.0,9.14,19.95,28.71,12.37,19.68,11.61,19.60,0.73,0.69,1.78
-1,1291.0,2.65,2.70,5.97,2.88,3.94,0.36,3.07,0.67,0.78,4.03
0,3188.0,2.66,2.13,4.25,2.22,2.55,0.01,1.28,0.56,0.62,7.47
1,1299.0,3.36,3.89,7.02,3.48,4.13,0.71,3.26,0.61,0.64,5.85
2,238.0,6.16,2.19,9.63,8.36,6.26,-1.10,4.17,0.24,0.37,10.08
